# Prompt Switch Qualitative Analysis (Video + AND Filter)

This notebook compares `baseline (CAFA)` vs `proposed (MMAudio)` using `__eval_filtered` metrics.

Supported criteria:
- `clap_classifier_flip`: CAFA `0` -> Proposed `1`
- `delta_flam_flip`: CAFA `< 0` -> Proposed `> 0`
- `desync_improved`: Proposed DeSync lower than CAFA
- `onset_f1_improved`: Proposed Onset F1 higher than CAFA
- `clap_score_improved`: Proposed CLAP target score higher than CAFA

You can combine multiple criteria with **AND** using widget multi-select.
Baseline video uses pre-generated CAFA mp4 files (no on-the-fly rendering).

In [ ]:
from pathlib import Path
from functools import lru_cache
from typing import Dict, List, Tuple, Optional, Sequence
import csv
import json
import shutil
import subprocess
import sys

from IPython.display import Video, Markdown, display

print('Python:', sys.version.split()[0])
print('Executable:', sys.executable)
print('ffmpeg:', shutil.which('ffmpeg'))

In [ ]:
BASELINE_ROOT = Path('/home/lgbin81/prompt_switch/CAFA/eval_vggsound_sparse_output/baseline_full')
PROPOSED_ROOT = Path('/home/lgbin81/prompt_switch/MMAudio/eval_vggsound_sparse_output/ablation_ts_17_sigma_0.0_method2/')
GT_VIDEO_ROOT = Path('/media/daftpunk5/dataset/vggsound/video')
GT_AUDIO_CACHE = Path('/home/lgbin81/prompt_switch/openFLAM/.cache/extracted_audio')

BASELINE_EVAL = BASELINE_ROOT / '__eval_filtered'
PROPOSED_EVAL = PROPOSED_ROOT / '__eval_filtered'


for p in [BASELINE_ROOT, PROPOSED_ROOT, GT_VIDEO_ROOT, BASELINE_EVAL, PROPOSED_EVAL]:
    print(f'{p}:', 'OK' if p.exists() else 'MISSING')


In [ ]:
def to_float(x: Optional[str], default: float = 0.0) -> float:
    try:
        return float(x)
    except Exception:
        return default


def load_csv_rows(path: Path) -> List[Dict[str, str]]:
    with path.open('r', encoding='utf-8') as f:
        return list(csv.DictReader(f))


def load_wrong_metrics(eval_dir: Path) -> Dict[Tuple[str, str], Dict]:
    rows = load_csv_rows(eval_dir / 'wrong_per_sample.csv')
    out = {}
    for r in rows:
        key = (r['video_id'], r['target_category'])
        clap_target = to_float(r.get('clap_score_target'))
        clap_source = to_float(r.get('clap_score_source'))
        out[key] = {
            'video_id': r['video_id'],
            'target_prompt': r['target_category'],
            'source_prompt': r.get('source_prompt', ''),
            'clap_classifier': int(to_float(r.get('clap_classifier'))),
            'clap_score_target': clap_target,
            'clap_score_source': clap_source,
            'clap_margin': clap_target - clap_source,
            'desync': to_float(r.get('desync')),
        }
    return out


def load_flam_metrics(eval_dir: Path) -> Dict[Tuple[str, str], Dict]:
    rows = load_csv_rows(eval_dir / 'flam_scores.csv')
    out = {}
    for r in rows:
        key = (r['sample_id'], r['target_prompt'])
        out[key] = {
            'delta_flam': to_float(r.get('delta_flam')),
            'onset_f1': to_float(r.get('onset_f1')),
        }
    return out


baseline_wrong = load_wrong_metrics(BASELINE_EVAL)
proposed_wrong = load_wrong_metrics(PROPOSED_EVAL)
baseline_flam = load_flam_metrics(BASELINE_EVAL)
proposed_flam = load_flam_metrics(PROPOSED_EVAL)

common_keys = sorted(set(baseline_wrong) & set(proposed_wrong))
print('Common samples:', len(common_keys))

records: List[Dict] = []
for k in common_keys:
    b = baseline_wrong[k]
    p = proposed_wrong[k]
    bf = baseline_flam.get(k, {})
    pf = proposed_flam.get(k, {})

    records.append({
        'video_id': k[0],
        'target_prompt': k[1],
        'source_prompt': p.get('source_prompt') or b.get('source_prompt', ''),

        'baseline_classifier': b['clap_classifier'],
        'proposed_classifier': p['clap_classifier'],

        'baseline_clap_target': b['clap_score_target'],
        'proposed_clap_target': p['clap_score_target'],
        'clap_target_gain': p['clap_score_target'] - b['clap_score_target'],

        'baseline_margin': b['clap_margin'],
        'proposed_margin': p['clap_margin'],
        'margin_gain': p['clap_margin'] - b['clap_margin'],

        'baseline_desync': b['desync'],
        'proposed_desync': p['desync'],
        'desync_gain': b['desync'] - p['desync'],

        'baseline_delta_flam': bf.get('delta_flam', 0.0),
        'proposed_delta_flam': pf.get('delta_flam', 0.0),
        'delta_flam_gain': pf.get('delta_flam', 0.0) - bf.get('delta_flam', 0.0),

        'baseline_onset_f1': bf.get('onset_f1', 0.0),
        'proposed_onset_f1': pf.get('onset_f1', 0.0),
        'onset_f1_gain': pf.get('onset_f1', 0.0) - bf.get('onset_f1', 0.0),
    })

print('Merged rows:', len(records))

In [ ]:
# Criteria + AND filter
CRITERIA_CONFIG = {
    'clap_classifier_flip': {
        'label': 'CLAP classifier flip (0 -> 1)',
        'min_gain': 0.0,
    },
    'delta_flam_flip': {
        'label': 'Delta FLAM flip (<0 -> >0)',
        'min_gain': 0.0,
    },
    'desync_improved': {
        'label': 'DeSync improved (gain > min)',
        'min_gain': 0.0,
    },
    'onset_f1_improved': {
        'label': 'Onset F1 improved (gain > min)',
        'min_gain': 0.0,
    },
    'clap_score_improved': {
        'label': 'CLAP target score improved (gain > min)',
        'min_gain': 0.0,
    },
}


def criterion_match(rec: Dict, criterion: str, min_gain: float = 0.0) -> bool:
    if criterion == 'clap_classifier_flip':
        return rec['baseline_classifier'] == 0 and rec['proposed_classifier'] == 1
    if criterion == 'delta_flam_flip':
        return rec['baseline_delta_flam'] < 0.0 and rec['proposed_delta_flam'] > 0.0
    if criterion == 'desync_improved':
        return rec['desync_gain'] > min_gain
    if criterion == 'onset_f1_improved':
        return rec['onset_f1_gain'] > min_gain
    if criterion == 'clap_score_improved':
        return rec['clap_target_gain'] > min_gain
    raise ValueError(f'Unknown criterion: {criterion}')


def criterion_score(rec: Dict, criterion: str) -> float:
    # Used for sorting within selected rows.
    if criterion == 'clap_classifier_flip':
        return rec['margin_gain'] + rec['delta_flam_gain'] + rec['clap_target_gain']
    if criterion == 'delta_flam_flip':
        return rec['proposed_delta_flam'] + rec['delta_flam_gain']
    if criterion == 'desync_improved':
        return rec['desync_gain']
    if criterion == 'onset_f1_improved':
        return rec['onset_f1_gain']
    if criterion == 'clap_score_improved':
        return rec['clap_target_gain']
    return 0.0


def _pass_extra_filters(rec: Dict, filter_cfg: Dict) -> bool:
    # 1) max desync (lower is better)
    max_desync = filter_cfg.get('max_proposed_desync', None)
    if max_desync is not None and rec['proposed_desync'] > float(max_desync):
        return False

    # 2) target prompt filter (if selected)
    target_prompts = filter_cfg.get('target_prompts', None)
    if target_prompts:
        if rec['target_prompt'] not in set(target_prompts):
            return False

    # 3) min FLAM (interpreted as min proposed_delta_flam)
    min_flam = filter_cfg.get('min_proposed_delta_flam', None)
    if min_flam is not None and rec['proposed_delta_flam'] < float(min_flam):
        return False

    # 4) min Delta FLAM gain
    min_delta_flam_gain = filter_cfg.get('min_delta_flam_gain', None)
    if min_delta_flam_gain is not None and rec['delta_flam_gain'] < float(min_delta_flam_gain):
        return False

    return True


def select_samples_and(
    all_records: List[Dict],
    selected_criteria: Sequence[str],
    min_gain_override: Optional[Dict[str, float]] = None,
    top_k: Optional[int] = None,
    filter_cfg: Optional[Dict] = None,
) -> List[Dict]:
    selected_criteria = list(selected_criteria)
    if not selected_criteria:
        selected_criteria = list(CRITERIA_CONFIG.keys())

    min_gain_override = min_gain_override or {}
    filter_cfg = filter_cfg or {}

    def _min_gain(c: str) -> float:
        return min_gain_override.get(c, CRITERIA_CONFIG[c].get('min_gain', 0.0))

    rows = []
    for r in all_records:
        ok = True
        for c in selected_criteria:
            if not criterion_match(r, c, _min_gain(c)):
                ok = False
                break
        if not ok:
            continue

        if not _pass_extra_filters(r, filter_cfg):
            continue

        rows.append(r)

    rows = sorted(
        rows,
        key=lambda r: sum(criterion_score(r, c) for c in selected_criteria),
        reverse=True,
    )

    if top_k is not None:
        rows = rows[:top_k]
    return rows


single_counts = {
    c: len(select_samples_and(records, [c]))
    for c in CRITERIA_CONFIG
}
for c in CRITERIA_CONFIG:
    print(f"{c:22s} | {single_counts[c]}")


In [ ]:
TABLE_COLS = [
    'video_id', 'source_prompt', 'target_prompt',
    'baseline_classifier', 'proposed_classifier',
    'baseline_clap_target', 'proposed_clap_target', 'clap_target_gain',
    'baseline_margin', 'proposed_margin', 'margin_gain',
    'baseline_delta_flam', 'proposed_delta_flam', 'delta_flam_gain',
    'baseline_desync', 'proposed_desync', 'desync_gain',
    'baseline_onset_f1', 'proposed_onset_f1', 'onset_f1_gain',
]


def preview_and_filter(
    selected_criteria: Sequence[str],
    top_k: int = 30,
    min_gain_override: Optional[Dict[str, float]] = None,
    filter_cfg: Optional[Dict] = None,
):
    rows = select_samples_and(
        records,
        selected_criteria,
        min_gain_override=min_gain_override,
        top_k=top_k,
        filter_cfg=filter_cfg,
    )
    print(f'Selected rows: {len(rows)} (criteria AND = {list(selected_criteria)})')
    if filter_cfg:
        print('Extra filters:', filter_cfg)

    if not rows:
        return rows

    try:
        import pandas as pd
        df = pd.DataFrame(rows)
        display(df[TABLE_COLS])
    except Exception:
        for i, r in enumerate(rows[:20]):
            print(f"[{i}] {r['video_id']} | {r['target_prompt']} | clap_gain={r['clap_target_gain']:.3f} desync={r['proposed_desync']:.3f}")
    return rows


_ = preview_and_filter(
    ['clap_classifier_flip', 'desync_improved'],
    top_k=20,
    filter_cfg={
        'max_proposed_desync': 10.0,
        'min_proposed_delta_flam': -1.0,
        'min_delta_flam_gain': -10.0,
    },
)


In [ ]:
# Media helpers

def normalize_text(s: str) -> str:
    return ' '.join(str(s).strip().lower().split())


def safe_slug(s: str) -> str:
    out = []
    for ch in s.lower().strip():
        if ch.isalnum() or ch in ('-', '_'):
            out.append(ch)
        elif ch.isspace():
            out.append('_')
        else:
            out.append('_')
    return ''.join(out)


@lru_cache(maxsize=4096)
def load_wrong_metadata(root_str: str, video_id: str) -> Dict:
    root = Path(root_str)
    md_path = root / video_id / 'wrong' / 'metadata.json'
    if not md_path.exists():
        return {}
    with md_path.open('r', encoding='utf-8') as f:
        return json.load(f)


def find_wrong_entry(root: Path, video_id: str, target_prompt: str) -> Optional[Dict]:
    md = load_wrong_metadata(str(root), video_id)
    target_norm = normalize_text(target_prompt)

    for item in md.get('results', []):
        if item.get('target_prompt') == target_prompt:
            return item

    for item in md.get('results', []):
        if normalize_text(item.get('target_prompt', '')) == target_norm:
            return item

    return None


def find_wrong_wav(root: Path, video_id: str, target_prompt: str) -> Optional[Path]:
    wrong_dir = root / video_id / 'wrong'
    entry = find_wrong_entry(root, video_id, target_prompt)

    if entry and entry.get('audio_file'):
        p = wrong_dir / entry['audio_file']
        if p.exists():
            return p

    fallback = wrong_dir / f"{video_id}_{target_prompt.replace(' ', '_')}.wav"
    return fallback if fallback.exists() else None


def find_wrong_mp4(root: Path, video_id: str, target_prompt: str) -> Optional[Path]:
    wrong_dir = root / video_id / 'wrong'
    entry = find_wrong_entry(root, video_id, target_prompt)

    if entry and entry.get('video_file'):
        p = wrong_dir / entry['video_file']
        if p.exists():
            return p

    fallback = wrong_dir / f"{video_id}_{target_prompt.replace(' ', '_')}.mp4"
    return fallback if fallback.exists() else None


def ensure_baseline_video(video_id: str, target_prompt: str) -> Optional[Path]:
    # No on-the-fly rendering: use pre-generated baseline mp4 only.
    baseline_mp4 = find_wrong_mp4(BASELINE_ROOT, video_id, target_prompt)
    if baseline_mp4 and baseline_mp4.exists():
        return baseline_mp4
    return None


In [ ]:
def show_sample_video(rows: Sequence[Dict], idx: int = 0, width: int = 560):
    if not rows:
        print('No rows to show.')
        return

    if idx < 0 or idx >= len(rows):
        raise IndexError(f'idx={idx} out of range (0 ~ {len(rows)-1})')

    r = rows[idx]

    header = (
        f"### {idx}/{len(rows)-1}: {r['video_id']}\n"
        f"- source: `{r['source_prompt']}`\n"
        f"- target: `{r['target_prompt']}`\n"
        f"- classifier: {r['baseline_classifier']} -> {r['proposed_classifier']}\n"
        f"- CLAP target: {r['baseline_clap_target']:.4f} -> {r['proposed_clap_target']:.4f} (gain={r['clap_target_gain']:.4f})\n"
        f"- margin: {r['baseline_margin']:.4f} -> {r['proposed_margin']:.4f} (gain={r['margin_gain']:.4f})\n"
        f"- delta_flam: {r['baseline_delta_flam']:.4f} -> {r['proposed_delta_flam']:.4f} (gain={r['delta_flam_gain']:.4f})\n"
        f"- desync: {r['baseline_desync']:.4f} -> {r['proposed_desync']:.4f} (gain={r['desync_gain']:.4f})\n"
        f"- onset_f1: {r['baseline_onset_f1']:.4f} -> {r['proposed_onset_f1']:.4f} (gain={r['onset_f1_gain']:.4f})"
    )
    display(Markdown(header))

    proposed_mp4 = find_wrong_mp4(PROPOSED_ROOT, r['video_id'], r['target_prompt'])
    baseline_video = ensure_baseline_video(r['video_id'], r['target_prompt'])

    display(Markdown('**Proposed video**'))
    if proposed_mp4 and proposed_mp4.exists():
        print(proposed_mp4)
        display(Video(filename=str(proposed_mp4), embed=True, width=width))
    else:
        print('Missing proposed mp4')

    display(Markdown('**Baseline video (pre-generated mp4)**'))
    if baseline_video and baseline_video.exists():
        print(baseline_video)
        display(Video(filename=str(baseline_video), embed=True, width=width))
    else:
        print('Missing baseline mp4')


In [ ]:
# Interactive widget with AND criteria (manual render; no auto-observe)
# Added filters:
# - max proposed desync
# - target prompt selection (Autocomplete)
# - min FLAM (proposed_delta_flam)
# - min Delta FLAM gain
try:
    import ipywidgets as widgets

    if '_psw_video_widgets' in globals():
        for _w in globals().get('_psw_video_widgets', []):
            try:
                _w.close()
            except Exception:
                pass
        globals()['_psw_video_widgets'] = []

    criterion_options = [(cfg['label'], key) for key, cfg in CRITERIA_CONFIG.items()]
    all_target_prompts = sorted({r['target_prompt'] for r in records})

    criteria_ms = widgets.SelectMultiple(
        options=criterion_options,
        value=('clap_classifier_flip',),
        description='AND criteria',
        rows=5,
    )

    target_prompt_ac = widgets.Combobox(
        value='',
        placeholder='Leave empty = ALL targets',
        options=all_target_prompts,
        description='target',
        ensure_option=False,
        continuous_update=False,
    )

    # criteria gain thresholds (for criteria that use gain)
    min_onset_gain = widgets.FloatText(value=0.0, description='min onset_f1 gain')
    min_clap_gain = widgets.FloatText(value=0.0, description='min clap gain')

    # extra global filters
    max_desync = widgets.FloatText(value=10.0, description='max desync')
    min_flam = widgets.FloatText(value=-1.0, description='min FLAM')
    min_delta_flam = widgets.FloatText(value=-10.0, description='min ΔFLAM')

    top_k_w = widgets.IntText(value=300, description='top_k')

    apply_btn = widgets.Button(description='Apply Filter', button_style='primary')
    render_btn = widgets.Button(description='Render Current', button_style='info')

    idx_slider = widgets.IntSlider(
        value=0, min=0, max=0, step=1, description='idx', continuous_update=False
    )

    out = widgets.Output()

    state = {
        'rows': [],
        'rendering': False,
    }

    def _normalize_target_value(v: str):
        t = (v or '').strip()
        if not t:
            return []

        # Exact match preferred
        if t in all_target_prompts:
            return [t]

        # Fallback: case-insensitive exact match
        low = t.lower()
        for x in all_target_prompts:
            if x.lower() == low:
                return [x]

        # If not found, keep empty (ALL) and show a hint in output
        return []

    def _build_rows():
        selected = list(criteria_ms.value)

        min_gain = {
            'onset_f1_improved': float(min_onset_gain.value),
            'clap_score_improved': float(min_clap_gain.value),
        }

        selected_targets = _normalize_target_value(target_prompt_ac.value)

        filter_cfg = {
            'max_proposed_desync': float(max_desync.value),
            'target_prompts': selected_targets,
            'min_proposed_delta_flam': float(min_flam.value),
            'min_delta_flam_gain': float(min_delta_flam.value),
        }

        tk = int(top_k_w.value) if int(top_k_w.value) > 0 else None
        return select_samples_and(
            records,
            selected,
            min_gain_override=min_gain,
            top_k=tk,
            filter_cfg=filter_cfg,
        )

    def _render_current(_=None):
        if state['rendering']:
            return

        state['rendering'] = True
        try:
            rows = state['rows']
            with out:
                out.clear_output(wait=False)
                selected = list(criteria_ms.value)
                selected_targets = _normalize_target_value(target_prompt_ac.value)
                target_label = selected_targets[0] if selected_targets else 'ALL'

                display(Markdown(
                    f"**Selected (AND):** `{selected}` | rows={len(rows)}\n"
                    f"**max_desync={max_desync.value}, min_FLAM={min_flam.value}, min_ΔFLAM={min_delta_flam.value}**\n"
                    f"**target:** `{target_label}`"
                ))

                if target_prompt_ac.value.strip() and not selected_targets:
                    print('Warning: target not found in options, using ALL targets.')

                if not rows:
                    print('No rows for current AND filter.')
                    return

                i = int(idx_slider.value)
                if i < 0:
                    i = 0
                if i >= len(rows):
                    i = len(rows) - 1

                show_sample_video(rows, idx=i)
        finally:
            state['rendering'] = False

    def _apply_filter(_=None):
        rows = _build_rows()
        state['rows'] = rows

        new_max = max(len(rows) - 1, 0)
        idx_slider.max = new_max
        if idx_slider.value > new_max:
            idx_slider.value = new_max
        if idx_slider.value < 0:
            idx_slider.value = 0

        _render_current()

    apply_btn.on_click(_apply_filter)
    render_btn.on_click(_render_current)

    controls0 = widgets.HBox([target_prompt_ac])
    controls1 = widgets.HBox([min_onset_gain, min_clap_gain, max_desync, min_flam, min_delta_flam, top_k_w])
    controls2 = widgets.HBox([apply_btn, render_btn, idx_slider])
    ui = widgets.VBox([criteria_ms, controls0, controls1, controls2, out])

    globals()['_psw_video_widgets'] = [
        criteria_ms, target_prompt_ac,
        min_onset_gain, min_clap_gain, max_desync, min_flam, min_delta_flam,
        top_k_w, apply_btn, render_btn, idx_slider, out, controls0, controls1, controls2, ui,
    ]

    display(ui)
    _apply_filter()

except Exception as e:
    print('ipywidgets not available. Use preview_and_filter(...) + show_sample_video(...) manually.')
    print('Reason:', e)


In [ ]:
# Mel-spectrogram comparison by video_id
# Usage:
#   compare_mel_by_video_id('qUnSjSA4kK0_000526')
#   compare_mel_by_video_id('qUnSjSA4kK0_000526', target_prompt='striking bowling')
#   compare_mel_by_video_id('qUnSjSA4kK0_000526', mode='correct')

import subprocess

import numpy as np
import matplotlib.pyplot as plt
import torchaudio
import torch


def _find_correct_wav(root: Path, video_id: str) -> Optional[Path]:
    correct_dir = root / video_id / 'correct'
    if not correct_dir.exists():
        return None

    exact = correct_dir / f'{video_id}_correct.wav'
    if exact.exists():
        return exact

    wavs = sorted(correct_dir.glob('*.wav'))
    return wavs[0] if wavs else None


def _find_gt_video(video_id: str) -> Optional[Path]:
    candidates = [
        GT_VIDEO_ROOT / f'{video_id}.mp4',
        GT_VIDEO_ROOT / f'{video_id}.wav',
        GT_VIDEO_ROOT.parent / 'video' / f'{video_id}.mp4',
        GT_VIDEO_ROOT.parent / 'video' / f'{video_id}.wav',
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    return None


def _ensure_gt_audio(video_id: str, target_sr: int = 16000) -> Optional[Path]:
    media_path = _find_gt_video(video_id)
    if media_path is None:
        return None

    if media_path.suffix.lower() == '.wav':
        return media_path

    GT_AUDIO_CACHE.mkdir(parents=True, exist_ok=True)
    cache_path = GT_AUDIO_CACHE / f'{video_id}_sr{target_sr}_mono.wav'
    if cache_path.exists():
        return cache_path

    subprocess.run(
        [
            'ffmpeg',
            '-y',
            '-i',
            str(media_path),
            '-vn',
            '-ac',
            '1',
            '-ar',
            str(target_sr),
            str(cache_path),
        ],
        check=True,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.PIPE,
        text=True,
    )
    return cache_path


def _available_wrong_targets(video_id: str) -> List[str]:
    b_md = load_wrong_metadata(str(BASELINE_ROOT), video_id)
    p_md = load_wrong_metadata(str(PROPOSED_ROOT), video_id)

    b_set = {x.get('target_prompt', '') for x in b_md.get('results', []) if x.get('target_prompt')}
    p_set = {x.get('target_prompt', '') for x in p_md.get('results', []) if x.get('target_prompt')}

    inter = sorted(b_set & p_set)
    if inter:
        return inter
    return sorted(b_set | p_set)


def _load_audio_mono(path: Path, target_sr: int = 16000) -> Tuple[torch.Tensor, int]:
    wav, sr = torchaudio.load(str(path))
    wav = wav.mean(dim=0)
    if sr != target_sr:
        wav = torchaudio.functional.resample(wav, sr, target_sr)
        sr = target_sr
    return wav, sr


def _mel_db(wav: torch.Tensor, sr: int, n_mels: int = 128, n_fft: int = 1024, hop: int = 256, fmax: int = 8000) -> np.ndarray:
    mel = torchaudio.transforms.MelSpectrogram(
        sample_rate=sr,
        n_fft=n_fft,
        hop_length=hop,
        n_mels=n_mels,
        f_min=0.0,
        f_max=fmax,
        power=2.0,
    )(wav.unsqueeze(0))
    mel_db = torchaudio.transforms.AmplitudeToDB(stype='power', top_db=80)(mel)
    return mel_db.squeeze(0).cpu().numpy()


def compare_mel_by_video_id(
    video_id: str,
    target_prompt: Optional[str] = None,
    mode: str = 'wrong',   # 'wrong' or 'correct'
    target_sr: int = 16000,
    n_mels: int = 128,
):
    if mode not in ('wrong', 'correct'):
        raise ValueError("mode must be 'wrong' or 'correct'")

    gt_wav = _ensure_gt_audio(video_id, target_sr=target_sr)
    if gt_wav is None or not gt_wav.exists():
        print('Missing GT audio extracted from source video for:', video_id)
        return

    if mode == 'wrong':
        targets = _available_wrong_targets(video_id)
        if not targets:
            print(f'No wrong targets found for video_id={video_id}')
            return

        if target_prompt is None:
            target_prompt = targets[0]

        if target_prompt not in targets:
            print('Requested target_prompt not found in shared target list.')
            print('Available targets:', targets)
            return

        baseline_wav = find_wrong_wav(BASELINE_ROOT, video_id, target_prompt)
        proposed_wav = find_wrong_wav(PROPOSED_ROOT, video_id, target_prompt)
        title_suffix = f'wrong | target={target_prompt}'
    else:
        baseline_wav = _find_correct_wav(BASELINE_ROOT, video_id)
        proposed_wav = _find_correct_wav(PROPOSED_ROOT, video_id)
        title_suffix = 'correct'

    if baseline_wav is None or not baseline_wav.exists():
        print('Missing baseline wav:', baseline_wav)
        return
    if proposed_wav is None or not proposed_wav.exists():
        print('Missing proposed wav:', proposed_wav)
        return

    gwav, gsr = _load_audio_mono(gt_wav, target_sr=target_sr)
    bwav, bsr = _load_audio_mono(baseline_wav, target_sr=target_sr)
    pwav, psr = _load_audio_mono(proposed_wav, target_sr=target_sr)

    gmel = _mel_db(gwav, gsr, n_mels=n_mels)
    bmel = _mel_db(bwav, bsr, n_mels=n_mels)
    pmel = _mel_db(pwav, psr, n_mels=n_mels)

    vmin = float(min(gmel.min(), bmel.min(), pmel.min()))
    vmax = float(max(gmel.max(), bmel.max(), pmel.max()))

    fig, axes = plt.subplots(3, 1, figsize=(14, 11), sharex=False)

    gdur = gwav.shape[0] / gsr
    bdur = bwav.shape[0] / bsr
    pdur = pwav.shape[0] / psr

    im0 = axes[0].imshow(gmel, origin='lower', aspect='auto', cmap='magma', vmin=vmin, vmax=vmax,
                         extent=[0, gdur, 0, gsr // 2])
    axes[0].set_title(f'GT (source video audio) mel-spectrogram | {video_id} | {title_suffix}')
    axes[0].set_ylabel('Freq (Hz)')
    axes[0].set_xlabel('Time (s)')

    im1 = axes[1].imshow(bmel, origin='lower', aspect='auto', cmap='magma', vmin=vmin, vmax=vmax,
                         extent=[0, bdur, 0, bsr // 2])
    axes[1].set_title(f'Baseline (CAFA) mel-spectrogram | {video_id} | {title_suffix}')
    axes[1].set_ylabel('Freq (Hz)')
    axes[1].set_xlabel('Time (s)')

    im2 = axes[2].imshow(pmel, origin='lower', aspect='auto', cmap='magma', vmin=vmin, vmax=vmax,
                         extent=[0, pdur, 0, psr // 2])
    axes[2].set_title(f'Proposed (MMAudio) mel-spectrogram | {video_id} | {title_suffix}')
    axes[2].set_ylabel('Freq (Hz)')
    axes[2].set_xlabel('Time (s)')

    cbar = fig.colorbar(im2, ax=axes, shrink=0.95)
    cbar.set_label('dB')

    print('GT source :', _find_gt_video(video_id))
    print('GT wav    :', gt_wav)
    print('Baseline  :', baseline_wav)
    print('Proposed  :', proposed_wav)
    if mode == 'wrong':
        print('Available targets:', targets)

    plt.tight_layout()
    plt.show()


# Example
compare_mel_by_video_id('9QwaP-cvdeU_000360', 'playing badminton')
